# Zinc-finger motif analysis of *C. elegans* tandem repeats

This notebook contains only the validated ZNF/PWM analysis. Genome extraction and Tandem Repeats Finder (TRF) execution are intentionally excluded: those steps are documented in the separate genome/TRF notebook.

## Workflow
1. Load the validated `TR_CATALOG_RAW.tsv` catalogue and recover the 27 candidate TR records. Input file: TR_CATALOG_RAW.tsv. If this file is not already available, it can be generated by running the TR.ipynb notebook first. Once generated, upload TR_CATALOG_RAW.tsv when prompted.
2. Resolve the ambiguous record and retain the 26 validated TR sequences.
3. Retrieve the 51 relevant CeMotifs PWM matrices and their factor annotations.
4. Scan TR consensus sequences and full repeat arrays for PWM matches and exact consensus sites.
5. Compare PRE50 sequences with the same PWM assigned to each corresponding TR.
6. Score individual monomers and characterize the below-threshold group.

**Interpretation:** PWM scores quantify sequence compatibility with a motif model. They do not demonstrate in-vivo protein binding.

## 1. Load the validated TR catalogue

Upload the previously validated `TR_CATALOG_RAW.tsv`. No genome or TRF processing is performed here.

In [1]:
from google.colab import files
from pathlib import Path
import io
import pandas as pd
import numpy as np
import zipfile
import urllib.request

uploaded = files.upload()
raw_names = [name for name in uploaded if "TR_CATALOG_RAW" in name]
if len(raw_names) != 1:
    raise RuntimeError(
        "Upload exactly one validated TR_CATALOG_RAW.tsv file. "
        f"Matching files found: {raw_names}"
    )

df_originale = pd.read_csv(io.BytesIO(uploaded[raw_names[0]]), sep="\t")
required_columns = {
    "Chromosome", "Start", "End", "Period", "Consensus", "Repeat_Sequence"
}
missing = required_columns - set(df_originale.columns)
if missing:
    raise ValueError(f"Missing required columns in RAW catalogue: {sorted(missing)}")

for col in ["Start", "End", "Period"]:
    df_originale[col] = pd.to_numeric(df_originale[col], errors="coerce")

print("RAW catalogue:", df_originale.shape)
print("Validated columns:", sorted(required_columns))

Saving TR_CATALOG_RAW.tsv to TR_CATALOG_RAW.tsv
RAW catalogue: (39098, 16)
Validated columns: ['Chromosome', 'Consensus', 'End', 'Period', 'Repeat_Sequence', 'Start']


## 2. Recover and validate the 27 candidate TR records

The first five AB identifiers refer to row indices in the validated RAW catalogue. The remaining identifiers are resolved by exact chromosome/start/end coordinates. The ambiguous AB record is checked separately and excluded from the 26-record analysis set.

In [ ]:
ID_MONOMERI = [
    "AB_1745", "AB_21010", "AB_32622", "AB_32946", "AB_33142",
    "MS_463", "MS_6476", "MS_6493", "MS_33627", "MS_29900",
    "C_1130", "C_12721", "C_12801", "C_15368", "C_10956",
    "D_18513", "D_32101", "E_10203", "E_19317",
    "AB_APO_X_4031472_4056451",
    "AB_IV_6682656_6731929",
    "AB_III_13584282_13599363",
    "AB_III_13584328_13599377",
    "AB_IV_1902960_1903281",
    "MS_III_1239493_1240806",
    "C_I_355869_356717",
    "DE_IV_6682656_6704725"
]

recovered = []
for identifier in ID_MONOMERI:
    parts = identifier.split("_")
    if len(parts) == 2 and parts[1].isdigit():
        raw_index = int(parts[1])
        if raw_index >= len(df_originale):
            recovered.append({"ID": identifier, "Numero_corrispondenze": 0})
            continue
        row = df_originale.iloc[raw_index]
        recovered.append({
            "ID": identifier, "Metodo": "RAW row index",
            "Indice_RAW": raw_index, "Numero_corrispondenze": 1,
            **row.to_dict()
        })
    else:
        chromosome, start, end = parts[-3], int(parts[-2]), int(parts[-1])
        hits = df_originale[
            (df_originale["Chromosome"].astype(str) == chromosome) &
            (df_originale["Start"] == start) &
            (df_originale["End"] == end)
        ]
        if len(hits) == 1:
            row = hits.iloc[0]
            recovered.append({
                "ID": identifier, "Metodo": "Exact genomic coordinates",
                "Indice_RAW": hits.index[0], "Numero_corrispondenze": 1,
                **row.to_dict()
            })
        else:
            recovered.append({
                "ID": identifier, "Metodo": "Multiple coordinate matches" if len(hits) > 1 else "Not found",
                "Numero_corrispondenze": len(hits)
            })

df_recupero = pd.DataFrame(recovered)
print("Candidate IDs:", len(ID_MONOMERI))
print("Unique recoveries:", int((df_recupero["Numero_corrispondenze"] == 1).sum()))
print("Ambiguous recoveries:", int((df_recupero["Numero_corrispondenze"] > 1).sum()))
print("Unresolved:", int((df_recupero["Numero_corrispondenze"] == 0).sum()))
display(df_recupero[["ID", "Metodo", "Indice_RAW", "Numero_corrispondenze"]])

In [ ]:
# Resolve the previously identified ambiguous AB record by its exact consensus.
AMBIGUOUS_ID = "AB_III_13584282_13599363"
ambiguous_consensus = "CCGCCGGTAGATTTTT" * 3

ambiguous_hits = df_originale[
    (df_originale["Chromosome"].astype(str) == "III") &
    (df_originale["Start"] == 13584282) &
    (df_originale["End"] == 13599363)
].copy()
ambiguous_hits["MATCH_CONSENSUS"] = (
    ambiguous_hits["Consensus"].astype(str) == ambiguous_consensus
)
display(ambiguous_hits[["Period", "Copy_Number", "Consensus", "MATCH_CONSENSUS"]])
assert int(ambiguous_hits["MATCH_CONSENSUS"].sum()) == 1, (
    "The ambiguous AB record was not resolved uniquely."
)
print("The ambiguous record is excluded from the 26-TR analysis set.")

In [ ]:
df_TR_ZF = df_recupero[
    (df_recupero["Numero_corrispondenze"] == 1) &
    (df_recupero["ID"] != AMBIGUOUS_ID)
].copy()

df_TR_ZF = df_TR_ZF[[
    "ID", "Indice_RAW", "Chromosome", "Start", "End", "Period",
    "Consensus", "Repeat_Sequence"
]].rename(columns={"Chromosome": "Cromosoma"}).reset_index(drop=True)

for column in ["Consensus", "Repeat_Sequence"]:
    df_TR_ZF[column] = df_TR_ZF[column].fillna("").astype(str).str.upper()
    df_TR_ZF[f"Valid_{column}"] = df_TR_ZF[column].str.fullmatch("[ACGT]+")
    df_TR_ZF[f"Length_{column}"] = df_TR_ZF[column].str.len()

assert len(df_TR_ZF) == 26, f"Expected 26 validated TRs, found {len(df_TR_ZF)}"
assert df_TR_ZF["ID"].nunique() == 26
assert df_TR_ZF["Valid_Consensus"].all()
assert df_TR_ZF["Valid_Repeat_Sequence"].all()

print("Validated TR records:", len(df_TR_ZF))
print("Consensus sequences:", int(df_TR_ZF["Valid_Consensus"].sum()))
print("Repeat sequences:", int(df_TR_ZF["Valid_Repeat_Sequence"].sum()))
display(df_TR_ZF[["ID", "Indice_RAW", "Cromosoma", "Start", "End", "Period",
                  "Length_Consensus", "Length_Repeat_Sequence"]])
df_scan_ZF = df_TR_ZF[["ID", "Indice_RAW", "Consensus"]].copy()
df_scan_ZF["Length"] = df_scan_ZF["Consensus"].str.len()
display(df_scan_ZF[["ID", "Indice_RAW", "Length"]].sort_values("Length"))